In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 40)

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
events = pd.read_csv('../data/events.csv',
                     parse_dates=['event_timestamp'])
orders = pd.read_csv('../data/orders.csv',
                     parse_dates=['order_created_at'])
ab     = pd.read_csv('../data/ab_assignments.csv',
                     parse_dates=['assigned_at','first_exposure_at'])

EXP_START = pd.Timestamp('2024-01-08')

print(f"events.csv         {len(events):>8,} rows")
print(f"orders.csv         {len(orders):>8,} rows")
print(f"ab_assignments.csv {len(ab):>8,} rows")
print(f"\nEvent columns: {events.columns.tolist()}")
print(f"\nNew columns present:")
print(f"  intent_level values:   {events['intent_level'].unique().tolist()}")
print(f"  dropoff_reason values: {sorted(events['dropoff_reason'].unique().tolist())}")
print(f"  page_type values:      {events['page_type'].unique().tolist()}")
display(events.head(3))

events.csv           99,843 rows
orders.csv            2,223 rows
ab_assignments.csv    8,000 rows

Event columns: ['user_pseudo_id', 'session_id', 'device_category', 'traffic_source', 'traffic_medium', 'item_id', 'item_name', 'product_category', 'price_tier', 'item_price', 'intent_level', 'dropoff_reason', 'event_id', 'event_timestamp', 'event_name', 'page_type', 'page_path', 'category_viewed', 'scroll_depth', 'order_id', 'order_revenue']

New columns present:
  intent_level values:   ['very_high', 'low', 'very_low', 'high', 'medium']
  dropoff_reason values: ['ad_pdp_mismatch', 'comparison_intent', 'converted', 'decision_friction', 'distraction', 'impulse_faded', 'out_of_stock', 'price_barrier', 'price_shock', 'wrong_audience']
  page_type values:      ['home', 'plp', 'pdp', 'checkout', 'confirmation']


,user_pseudo_id,session_id,device_category,traffic_source,traffic_medium,item_id,item_name,product_category,price_tier,item_price,intent_level,dropoff_reason,event_id,event_timestamp,event_name,page_type,page_path,category_viewed,scroll_depth,order_id,order_revenue
0,uid_3791eb90,sess_uid_3791eb90_20230713121200,desktop,email,email,LUM-SH-03,Hydra Boost Shampoo 32oz,Shampoo,mid,39.99,very_high,decision_friction,evt_71c5e9e9,2023-07-13 12:12:12,page_view,home,/,NaN,0.59,NaN,NaN
1,uid_3791eb90,sess_uid_3791eb90_20230713121200,desktop,email,email,LUM-SH-03,Hydra Boost Shampoo 32oz,Shampoo,mid,39.99,very_high,decision_friction,evt_c41bf736,2023-07-13 12:12:51,view_item_list,plp,/collections/shampoo,Shampoo,0.51,NaN,NaN
2,uid_3791eb90,sess_uid_3791eb90_20230713121200,desktop,email,email,LUM-SH-03,Hydra Boost Shampoo 32oz,Shampoo,mid,39.99,very_high,decision_friction,evt_4c64477e,2023-07-13 12:14:05,view_item,pdp,/products/lum-sh-03,Shampoo,0.72,NaN,NaN


In [3]:
print("=" * 55)
print("DATA QUALITY CHECKS")
print("=" * 55)

# 1. Duplicate event IDs
print(f"\n1. Duplicate event_ids:        {events['event_id'].duplicated().sum()}")

# 2. Null user IDs
print(f"2. Null user_pseudo_id:        {events['user_pseudo_id'].isnull().sum()}")

# 3. Duplicate order IDs in orders table
print(f"3. Duplicate order_ids:        {orders['order_id'].duplicated().sum()}")

# 4. Orphan orders
purchase_order_ids = events[events['event_name']=='purchase']['order_id'].dropna()
orphans = ~orders['order_id'].isin(purchase_order_ids)
print(f"4. Orphan orders:              {orphans.sum()}")

# 5. Date range
print(f"\n5. Events date range:")
print(f"   Min: {events['event_timestamp'].min()}")
print(f"   Max: {events['event_timestamp'].max()}")

# 6. Full funnel event counts
print(f"\n6. Event breakdown:")
print(events['event_name'].value_counts().to_string())

# 7. Scroll depth only on PDP events
non_pdp_scroll = events[
    (events['event_name'] != 'view_item') &
    (events['scroll_depth'].notna())
]['event_name'].value_counts()
print(f"\n7. Scroll depth on non-PDP events (should be home/PLP only):")
print(non_pdp_scroll.to_string())

# 8. Intent level distribution
print(f"\n8. Intent level distribution:")
print(events['intent_level'].value_counts().to_string())

# 9. Drop-off reason distribution
print(f"\n9. Drop-off reason distribution:")
print(events['dropoff_reason'].value_counts().to_string())

print("\n✓ Quality checks complete")

DATA QUALITY CHECKS

1. Duplicate event_ids:        0
2. Null user_pseudo_id:        0
3. Duplicate order_ids:        0
4. Orphan orders:              0

5. Events date range:
   Min: 2023-04-01 00:16:03
   Max: 2024-04-06 23:54:09

6. Event breakdown:
event_name
page_view         45677
view_item_list    26027
view_item         18358
add_to_cart        4892
begin_checkout     2666
purchase           2223

7. Scroll depth on non-PDP events (should be home/PLP only):
event_name
page_view         45677
view_item_list    26027

8. Intent level distribution:
intent_level
very_high    36319
high         28843
low          25540
medium        4718
very_low      4423

9. Drop-off reason distribution:
dropoff_reason
converted            36404
decision_friction    12498
price_barrier         9812
comparison_intent     8480
price_shock           8436
impulse_faded         8418
distraction           6765
wrong_audience        5817
out_of_stock          2422
ad_pdp_mismatch        791

✓ Quality ch

In [4]:
pre_events = events[events['event_timestamp'] <  EXP_START].copy()
exp_events = events[events['event_timestamp'] >= EXP_START].copy()

print(f"Pre-experiment events:  {len(pre_events):>8,}")
print(f"Experiment events:      {len(exp_events):>8,}")
print(f"Total:                  {len(pre_events)+len(exp_events):>8,}")

print(f"\nPre-experiment funnel:")
print(pre_events['event_name'].value_counts().to_string())

print(f"\nExperiment funnel:")
print(exp_events['event_name'].value_counts().to_string())

Pre-experiment events:    63,659
Experiment events:        36,184
Total:                    99,843

Pre-experiment funnel:
event_name
page_view         29122
view_item_list    16614
view_item         11666
add_to_cart        3138
begin_checkout     1701
purchase           1418

Experiment funnel:
event_name
page_view         16555
view_item_list     9413
view_item          6692
add_to_cart        1754
begin_checkout      965
purchase            805


In [5]:
FUNNEL_STAGES = [
    'page_view',       # Home
    'view_item_list',  # PLP
    'view_item',       # PDP
    'add_to_cart',     # ATC
    'begin_checkout',  # Checkout
    'purchase',        # Purchase
]

def sessionize(df):
    """
    Aggregates raw events into one row per session.
    Handles the full 6-stage funnel:
    Home → PLP → PDP → ATC → Checkout → Purchase

    New vs old sessionize:
    - 6 funnel stages instead of 5 (added home + PLP)
    - intent_level and dropoff_reason per session
    - avg_scroll_depth from PDP view_item events
    - session_duration from first to last event
    """

    # ── Step 1: Count each event type per session ──────────────────
    event_counts = (
        df.groupby(['session_id','event_name'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    # Ensure all funnel columns exist
    for stage in FUNNEL_STAGES:
        if stage not in event_counts.columns:
            event_counts[stage] = 0

    # ── Step 2: Session-level attributes ───────────────────────────
    session_attrs = (
        df.groupby('session_id')
        .agg(
            user_pseudo_id    = ('user_pseudo_id',   'first'),
            session_start     = ('event_timestamp',  'min'),
            session_end       = ('event_timestamp',  'max'),
            device_category   = ('device_category',  'first'),
            traffic_source    = ('traffic_source',   'first'),
            traffic_medium    = ('traffic_medium',   'first'),
            item_id           = ('item_id',          'first'),
            product_category  = ('product_category', 'first'),
            price_tier        = ('price_tier',       'first'),
            item_price        = ('item_price',       'first'),
            intent_level      = ('intent_level',     'first'),
            dropoff_reason    = ('dropoff_reason',   'first'),
            category_viewed   = ('category_viewed',  'first'),
            order_id          = ('order_id',         'last'),
            order_revenue     = ('order_revenue',    'sum'),
        )
        .reset_index()
    )

    # ── Step 3: Avg scroll depth (PDP events only) ──────────────────
    pdp_scroll = (
        df[df['event_name']=='view_item']
        .groupby('session_id')['scroll_depth']
        .mean()
        .reset_index()
        .rename(columns={'scroll_depth':'avg_scroll_depth'})
    )

    # ── Step 4: Merge everything ───────────────────────────────────
    sessions = session_attrs.merge(
        event_counts[['session_id'] + FUNNEL_STAGES],
        on='session_id', how='left'
    )
    sessions = sessions.merge(pdp_scroll, on='session_id', how='left')

    # ── Step 5: Boolean funnel flags ───────────────────────────────
    sessions['reached_home']     = sessions['page_view']       > 0
    sessions['reached_plp']      = sessions['view_item_list']  > 0
    sessions['reached_pdp']      = sessions['view_item']       > 0
    sessions['reached_atc']      = sessions['add_to_cart']     > 0
    sessions['reached_checkout'] = sessions['begin_checkout']  > 0
    sessions['purchased']        = sessions['purchase']        > 0

    # ── Step 6: Derived features ───────────────────────────────────
    sessions['pdp_view_count']       = sessions['view_item']
    sessions['session_duration_sec'] = (
        sessions['session_end'] - sessions['session_start']
    ).dt.total_seconds()

    return sessions

pre_exp_sessions = sessionize(pre_events)
exp_sessions     = sessionize(exp_events)

print(f"Pre-experiment sessions: {len(pre_exp_sessions):,}")
print(f"Experiment sessions:     {len(exp_sessions):,}")
display(pre_exp_sessions.head(3))

Pre-experiment sessions: 29,122
Experiment sessions:     16,555


,session_id,user_pseudo_id,session_start,session_end,device_category,traffic_source,traffic_medium,item_id,product_category,price_tier,item_price,intent_level,dropoff_reason,category_viewed,order_id,order_revenue,page_view,view_item_list,view_item,add_to_cart,begin_checkout,purchase,avg_scroll_depth,reached_home,reached_plp,reached_pdp,reached_atc,reached_checkout,purchased,pdp_view_count,session_duration_sec
0,sess_uid_00020d4f_20230712061200,uid_00020d4f,2023-07-12 06:12:13,2023-07-12 06:12:13,mobile,instagram,paid_social,LUM-BDL-01,Bundle,mid,42.99,low,price_shock,None,None,0.00,1,0,0,0,0,0,NaN,True,False,False,False,False,False,0,0.00
1,sess_uid_00020d4f_20230725121200,uid_00020d4f,2023-07-25 12:12:18,2023-07-25 12:12:37,mobile,google,cpc,LUM-BDL-02,Bundle,mid,39.99,high,decision_friction,Bundle,None,0.00,1,1,0,0,0,0,NaN,True,True,False,False,False,False,0,19.00
2,sess_uid_00020d4f_20230822121200,uid_00020d4f,2023-08-22 12:12:17,2023-08-22 12:13:12,mobile,google,cpc,LUM-BDL-02,Bundle,mid,39.99,high,decision_friction,Bundle,None,0.00,1,1,0,0,0,0,NaN,True,True,False,False,False,False,0,55.00


In [6]:
print("=" * 62)
print("FUNNEL VALIDATION — PRE-EXPERIMENT")
print("=" * 62)

total    = len(pre_exp_sessions)
home     = pre_exp_sessions['reached_home'].sum()
plp      = pre_exp_sessions['reached_plp'].sum()
pdp      = pre_exp_sessions['reached_pdp'].sum()
atc      = pre_exp_sessions['reached_atc'].sum()
checkout = pre_exp_sessions['reached_checkout'].sum()
purchase = pre_exp_sessions['purchased'].sum()

stages  = ['Home','PLP','PDP','ATC','Checkout','Purchase']
volumes = [home, plp, pdp, atc, checkout, purchase]

print(f"\n{'STAGE':<12} {'VOLUME':>8}  {'% OF HOME':>10}  {'STEP CVR':>10}")
print("=" * 48)
for i,(stage,vol) in enumerate(zip(stages,volumes)):
    pct_home = vol/home*100 if home>0 else 0
    step_cvr = f"{vol/volumes[i-1]:.1%}" if i>0 and volumes[i-1]>0 else "—"
    print(f"{stage:<12} {vol:>8,}  {pct_home:>9.1f}%  {step_cvr:>10}")

print("=" * 48)
aov = pre_exp_sessions.loc[
    pre_exp_sessions['purchased']==True,'order_revenue'].mean()
print(f"\nAverage Order Value: ${aov:.2f}")
print(f"Overall CVR (home→purchase): {purchase/home:.2%}")

# Intent level distribution in sessions
print(f"\nIntent level distribution:")
print(pre_exp_sessions['intent_level'].value_counts().to_string())

# Drop-off reason distribution
print(f"\nDrop-off reason distribution:")
print(pre_exp_sessions['dropoff_reason'].value_counts().to_string())

# Scroll depth summary
print(f"\nAvg scroll depth (sessions with PDP view):")
pdp_sessions = pre_exp_sessions[pre_exp_sessions['reached_pdp']==True]
print(pdp_sessions['avg_scroll_depth'].describe().round(3).to_string())

FUNNEL VALIDATION — PRE-EXPERIMENT

STAGE          VOLUME   % OF HOME    STEP CVR
Home           29,122      100.0%           —
PLP            16,614       57.0%       57.0%
PDP             7,836       26.9%       47.2%
ATC             3,138       10.8%       40.0%
Checkout        1,701        5.8%       54.2%
Purchase        1,418        4.9%       83.4%

Average Order Value: $44.06
Overall CVR (home→purchase): 4.87%

Intent level distribution:
intent_level
low          11147
very_high     7274
high          7205
very_low      2085
medium        1411

Drop-off reason distribution:
dropoff_reason
converted            7710
price_shock          3777
impulse_faded        3743
price_barrier        2770
distraction          2687
decision_friction    2625
wrong_audience       2549
comparison_intent    2373
out_of_stock          643
ad_pdp_mismatch       245

Avg scroll depth (sessions with PDP view):
count   7836.00
mean       0.55
std        0.27
min        0.01
25%        0.26
50%        0

In [7]:
print("=" * 55)
print("NEW COLUMN VALIDATION")
print("=" * 55)

# 1. Intent level by traffic source — sanity check
print("\n1. Avg intent level by traffic source:")
intent_order = {'very_low':1,'low':2,'medium':3,'high':4,'very_high':5}
pre_exp_sessions['intent_score'] = pre_exp_sessions['intent_level'].map(intent_order)
intent_by_source = (
    pre_exp_sessions.groupby('traffic_source')['intent_score']
    .mean().sort_values(ascending=False)
)
print(intent_by_source.round(2).to_string())
print("  (Expected: direct/email highest, facebook/instagram lowest)")

# 2. Scroll depth by drop-off reason — sanity check
print("\n2. Avg scroll depth by drop-off reason (PDP sessions):")
scroll_by_reason = (
    pdp_sessions.groupby('dropoff_reason')['avg_scroll_depth']
    .mean().sort_values(ascending=False)
)
print(scroll_by_reason.round(3).to_string())
print("  (Expected: decision_friction highest, wrong_audience lowest)")

# 3. Session duration by drop-off reason
print("\n3. Avg session duration (sec) by drop-off reason:")
dur_by_reason = (
    pdp_sessions.groupby('dropoff_reason')['session_duration_sec']
    .mean().sort_values(ascending=False)
)
print(dur_by_reason.round(1).to_string())
print("  (Expected: decision_friction longest, impulse_faded shortest)")

# 4. PDP view count by drop-off reason
print("\n4. Avg PDP views by drop-off reason:")
views_by_reason = (
    pdp_sessions.groupby('dropoff_reason')['pdp_view_count']
    .mean().sort_values(ascending=False)
)
print(views_by_reason.round(2).to_string())
print("  (Expected: decision_friction/comparison_intent highest)")

NEW COLUMN VALIDATION

1. Avg intent level by traffic source:
traffic_source
email       4.73
direct      4.72
google      3.73
referral    3.70
tiktok      1.83
instagram   1.83
  (Expected: direct/email highest, facebook/instagram lowest)

2. Avg scroll depth by drop-off reason (PDP sessions):
dropoff_reason
decision_friction   0.79
converted           0.74
comparison_intent   0.63
distraction         0.37
out_of_stock        0.32
price_shock         0.23
price_barrier       0.18
impulse_faded       0.11
ad_pdp_mismatch     0.06
wrong_audience      0.03
  (Expected: decision_friction highest, wrong_audience lowest)

3. Avg session duration (sec) by drop-off reason:
dropoff_reason
decision_friction   379.20
converted           258.30
comparison_intent   197.20
distraction          93.40
impulse_faded        93.30
wrong_audience       93.20
out_of_stock         93.00
price_barrier        92.30
price_shock          89.60
ad_pdp_mismatch      85.00
  (Expected: decision_friction longest,

In [8]:
exp_sessions = exp_sessions.merge(
    ab[['user_pseudo_id','variant']],
    on='user_pseudo_id', how='left'
)

print("Variant distribution in experiment sessions:")
print(exp_sessions['variant'].value_counts().to_string())
print(f"\nSessions with no variant: {exp_sessions['variant'].isnull().sum()}")

# Validate intent distribution is balanced across variants
print("\nIntent level by variant (should be similar):")
intent_by_variant = (
    exp_sessions.groupby('variant')['intent_level']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(3)
)
print(intent_by_variant.to_string())

Variant distribution in experiment sessions:
variant
control      8372
treatment    8183

Sessions with no variant: 0

Intent level by variant (should be similar):
intent_level  high  low  medium  very_high  very_low
variant                                             
control       0.25 0.38    0.05       0.25      0.07
treatment     0.24 0.39    0.05       0.25      0.07


In [9]:
order_details = orders[['order_id','is_returned','payment_method']].copy()

pre_exp_sessions = pre_exp_sessions.merge(
    order_details, on='order_id', how='left')
exp_sessions     = exp_sessions.merge(
    order_details, on='order_id', how='left')

print("Pre-experiment sessions:")
print(f"  Total:          {len(pre_exp_sessions):,}")
print(f"  With purchases: {pre_exp_sessions['purchased'].sum():,}")
print(f"  With returns:   {pre_exp_sessions['is_returned'].notna().sum():,}")

print("\nExperiment sessions:")
print(f"  Total:          {len(exp_sessions):,}")
print(f"  With purchases: {exp_sessions['purchased'].sum():,}")
print(f"  With returns:   {exp_sessions['is_returned'].notna().sum():,}")

Pre-experiment sessions:
  Total:          29,122
  With purchases: 1,418
  With returns:   1,418

Experiment sessions:
  Total:          16,555
  With purchases: 805
  With returns:   805


In [10]:
print("=" * 55)
print("FINAL SCHEMA — PRE-EXPERIMENT SESSIONS")
print("=" * 55)

print(f"\nShape: {pre_exp_sessions.shape}")
print(f"\nColumns:")
for col in pre_exp_sessions.columns:
    dtype  = pre_exp_sessions[col].dtype
    nulls  = pre_exp_sessions[col].isnull().sum()
    sample = pre_exp_sessions[col].dropna().iloc[0] if len(
        pre_exp_sessions[col].dropna()) > 0 else "—"
    print(f"  {col:<30} {str(dtype):<12} nulls:{nulls:<6} sample:{sample}")

FINAL SCHEMA — PRE-EXPERIMENT SESSIONS

Shape: (29122, 34)

Columns:
  session_id                     object       nulls:0      sample:sess_uid_00020d4f_20230712061200
  user_pseudo_id                 object       nulls:0      sample:uid_00020d4f
  session_start                  datetime64[ns] nulls:0      sample:2023-07-12 06:12:13
  session_end                    datetime64[ns] nulls:0      sample:2023-07-12 06:12:13
  device_category                object       nulls:0      sample:mobile
  traffic_source                 object       nulls:0      sample:instagram
  traffic_medium                 object       nulls:0      sample:paid_social
  item_id                        object       nulls:0      sample:LUM-BDL-01
  product_category               object       nulls:0      sample:Bundle
  price_tier                     object       nulls:0      sample:mid
  item_price                     float64      nulls:0      sample:42.99
  intent_level                   object       nulls:0     

In [11]:
pre_exp_sessions.to_csv('../data/pre_exp_sessions.csv', index=False)
exp_sessions.to_csv    ('../data/exp_sessions.csv',     index=False)

print("Saved:")
print(f"  pre_exp_sessions.csv  {len(pre_exp_sessions):,} rows | {pre_exp_sessions.shape[1]} cols")
print(f"  exp_sessions.csv      {len(exp_sessions):,} rows | {exp_sessions.shape[1]} cols")

print(f"\nNew columns vs old version:")
new_cols = ['reached_home','reached_plp','intent_level',
            'dropoff_reason','avg_scroll_depth','category_viewed']
for col in new_cols:
    present = col in pre_exp_sessions.columns
    print(f"  {col:<30} {'✓ present' if present else '✗ missing'}")

print("\n✓ Data prep complete — ready for 02_funnel_analysis.ipynb")

Saved:
  pre_exp_sessions.csv  29,122 rows | 34 cols
  exp_sessions.csv      16,555 rows | 34 cols

New columns vs old version:
  reached_home                   ✓ present
  reached_plp                    ✓ present
  intent_level                   ✓ present
  dropoff_reason                 ✓ present
  avg_scroll_depth               ✓ present
  category_viewed                ✓ present

✓ Data prep complete — ready for 02_funnel_analysis.ipynb
